# GE Projection from 2026 Council Election Votes

This notebook takes the ward-level votes from today's UK council elections and projects
what a General Election held on the same day might have looked like in seat terms.
It uses the standard **PNS + UNS** model (see next cell).

**This is a thought experiment, not a prediction.** Council elections differ from GEs
in turnout, ballot dynamics, candidate effects, and salience. Read the seat totals
accordingly. The full caveats are listed at the bottom.

## What is PNS + UNS?

**PNS — Projected National Share.** The councils that voted today are not a
representative sample of GB. PNS estimates what the national vote share *would*
have been if the whole country had voted today, derived from the wards that did.
We use a simple unweighted aggregation across wards (the full Curtice technique
weights wards by historical patterns; see Caveats).

**UNS — Uniform National Swing.** A *swing* is the change in a party's vote share
vs the last GE. UNS applies the same percentage-point shift to every constituency:

> `projected_share[c, p] = max(0, GE2024_share[c, p] + swing[p])`

After applying, we renormalise within each constituency so shares sum to 100% and
pick the party with the highest projected share.

**Worked example.** If Lab's 2024 national share was 33.7% and our PNS for Lab is
21.2%, the Lab swing is −12.5pp. In a seat where Lab had 50% in 2024, our model
projects 37.5%. In a seat where Lab had 25%, our model projects 12.5%. Same shift
everywhere — that's the "uniform" part. UNS is simple and surprisingly accurate
as a first approximation, but it misses real-world non-uniformity (parties tend
to collapse hardest where they were strongest).

**Parties without a swing signal** (SNP, Plaid Cymru, NI parties, independents)
keep their 2024 per-constituency shares unchanged. We compute swings only for the
five GB-wide majors: RFM, CON, LAB, GRN, LDM.

## Setup

In [1]:
import sys
from pathlib import Path

# Make `src/` importable regardless of where Jupyter was launched.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import pandas as pd

from src.sources import load_council_results, load_ge2024_constituency
from src.clean import normalise_council_results
from src.pns import compute_pns
from src.swing import compute_national_2024, compute_swing, apply_uns
from src.project import pick_winners, seat_totals

pd.set_option("display.float_format", "{:.4f}".format)

## Load data

In [2]:
council_raw = load_council_results()
ge2024 = load_ge2024_constituency()

print(f"Council rows (raw):       {len(council_raw):,}")
print(f"GE2024 long rows:         {len(ge2024):,}")
print(f"GE2024 unique seats:      {ge2024['constituency_id'].nunique():,}")

Council rows (raw):       2,554
GE2024 long rows:         8,450
GE2024 unique seats:      650


## Clean council data

In [3]:
council_long = normalise_council_results(council_raw)
n_wards = council_long[['council', 'ward']].drop_duplicates().shape[0]
print(f"Wards with results: {n_wards:,}")
council_long.head(8)

Wards with results: 2,390


,council,ward,party,votes
0,Adur,Buckingham,RFM,393
1,Adur,Churchill,RFM,459
2,Adur,Cokeham,RFM,650
3,Adur,Eastbrook,RFM,379
4,Adur,Hillside,RFM,463
5,Adur,Manor,RFM,473
6,Adur,Marine,RFM,289
7,Adur,Mash Barn,RFM,598


## Compute PNS

Sanity check: this should land within ~0.5pp of the headline percentages at the
top of the source sheet (RFM 25.0%, LAB 21.2%, GRN 18.8%, CON 17.0%, LDM 12.7%).

In [4]:
pns = compute_pns(council_long)
pns_table = (pns * 100).round(2).rename("PNS (%)").to_frame()
pns_table

,PNS (%)
party,
RFM,24.1100
LAB,20.2100
GRN,18.2600
CON,17.6400
LDM,13.7000
OTH,6.0700


In [5]:
assert abs(pns.sum() - 1.0) < 1e-9, "PNS does not sum to 1.0"

## 2024 national shares and swings

In [6]:
ge_national = compute_national_2024(ge2024)
swing = compute_swing(pns, ge_national)

table = pd.DataFrame({
    "GE2024 (%)": (ge_national * 100).round(2),
    "PNS 2026 (%)": (pns.reindex(ge_national.index) * 100).round(2),
    "Swing (pp)": (swing * 100).round(2),
})
table

,GE2024 (%),PNS 2026 (%),Swing (pp)
party,,,
RFM,14.2900,24.1100,9.8200
CON,23.7000,17.6400,-6.0600
LAB,33.7000,20.2100,-13.4900
GRN,6.7500,18.2600,11.5100
LDM,12.2200,13.7000,1.4900


## Apply UNS to per-constituency 2024 shares

In [7]:
projected = apply_uns(ge2024, swing)

# Sanity: every constituency's projected shares sum to 1.0.
sums = projected.groupby("constituency_id")["projected_share"].sum()
assert (sums - 1.0).abs().max() < 1e-9, "renormalisation failed somewhere"
print(f"Per-constituency share sums: min={sums.min():.6f}, max={sums.max():.6f}")

Per-constituency share sums: min=1.000000, max=1.000000


### Sample constituencies (before vs after)

In [8]:
samples = [
    "Holborn and St Pancras",     # safe Lab (Starmer's seat)
    "Witney",                     # safe Con
    "Sheffield Hallam",           # Lab/LD marginal type
    "Glasgow North",              # Scotland (SNP share held flat)
    "Cardiff South and Penarth",  # Wales (PC share held flat)
]
for name in samples:
    print(f"\n--- {name} ---")
    seat = (
        projected.query("constituency_name == @name")
        .assign(share_pct=lambda d: (d["share"] * 100).round(2),
                proj_pct=lambda d: (d["projected_share"] * 100).round(2))
        [["party", "share_pct", "proj_pct"]]
        .sort_values("proj_pct", ascending=False)
        .head(6)
    )
    print(seat.to_string(index=False))


--- Holborn and St Pancras ---
party  share_pct  proj_pct
  LAB    48.9200   34.3100
  GRN    10.4400   21.2600
  OTH    21.5100   20.8300
  RFM     6.1400   15.4600
  LDM     5.7900    7.0500
  CON     7.1900    1.0900

--- Witney ---
party  share_pct  proj_pct
  LDM    41.1800   39.7600
  CON    32.6100   24.7300
  RFM    12.4700   20.7700
  GRN     3.2800   13.7900
  OTH     1.0200    0.9500
  LAB     9.4400    0.0000

--- Sheffield Hallam ---
party  share_pct  proj_pct
  LAB    46.2700   31.7400
  LDM    30.4000   30.8800
  GRN     8.7000   19.5700
  RFM     0.0000    9.5100
  CON    12.0200    5.7700
  OTH     2.6000    2.5200

--- Glasgow North ---
party  share_pct  proj_pct
  SNP    32.0000   30.3600
  LAB    42.1900   27.2300
  GRN    12.1900   22.4800
  RFM     4.7600   13.8400
  LDM     3.2900    4.5300
  OTH     1.6500    1.5600

--- Cardiff South and Penarth ---
party  share_pct  proj_pct
  LAB    44.4900   30.0200
  GRN    14.4500   25.1400
  RFM    11.4700   20.6200
  LD

## Headline result: projected seat totals vs 2024

In [9]:
totals = seat_totals(projected, ge2024)
totals

,party,projected_seats,ge2024_seats,change
0,LAB,218,409,-191
1,RFM,185,5,180
2,CON,95,121,-26
3,LDM,80,72,8
4,SNP,24,9,15
5,OTH,20,10,10
6,GRN,8,4,4
7,SF,7,7,0
8,DUP,5,5,0
9,PC,4,4,0


## Caveats — please read before quoting any of these numbers

1. **PNS is computed naively.** We aggregate ward vote shares with equal weights
   instead of the Curtice-style weighting that corrects for non-representative
   electing-council geography. If the electing councils lean systematically one way,
   our PNS is biased in that direction.
2. **Top-candidate votes are not party totals.** The source sheet records the
   highest-polling candidate per party in each ward. Parties fielding a full slate
   in multi-member wards have more total votes than this column shows, so they're
   slightly under-counted in PNS.
3. **Independents and local-group votes don't generate a swing.** They're in the
   PNS denominator (so the major-party shares are honest) and held flat per
   constituency in the projection. There's no clean signal for "the independent vote"
   as a coherent national bloc.
4. **UNS is a first approximation.** Real swings aren't uniform across regions or
   constituency types — a party that's collapsing tends to collapse harder where it
   was strongest. UNS misses this.
5. **Parties without a swing signal are held flat.** SNP, Plaid Cymru, NI parties,
   independents, minor parties all keep their 2024 per-constituency shares.
   Renormalisation within each constituency keeps shares summing to 100%.
6. **Parties new to a seat.** Where a party had no 2024 candidate, applying a
   positive swing produces a non-zero projected share — the model treats it as if
   they fielded a candidate.
7. **This is a model, not a prediction.** Council ballot dynamics differ from GE
   ballot dynamics. Read the seat totals as a thought experiment.